# 세션4 — Indexing 알고리즘 (간단 소개)

HNSW 같은 ANN(Approximate Nearest Neighbor) 인덱스의 개념은 PPT에서 다룹니다. 여기서는 코드로 봤을 때 **얼마나 간단히 쓸 수 있는지**, 그리고 **왜 오늘 우리 실습 데이터로는 효과가 안 보이는지**만 확인하고 넘어갑니다.

## Brute-force vs ANN(HNSW), 한 줄 요약

- **Brute-force(전수 비교)**: 질문 벡터를 저장된 모든 벡터와 하나씩 비교합니다. 벡터 개수(N)가 늘어나는 만큼 검색 시간도 그대로 늘어납니다.
- **ANN(예: HNSW)**: 벡터들 사이의 "그래프"를 미리 만들어두고, 검색할 때는 이 그래프를 타고 내려가며 후보를 빠르게 좁힙니다. 검색은 훨씬 빨라지지만, **그래프를 미리 만드는 비용**과 **약간의 부정확함(근사치)**을 대가로 치릅니다.

사실 우리가 지금까지 써온 `Chroma`는 내부적으로 이미 HNSW를 쓰고 있습니다. 즉 새로 배울 코드는 거의 없고, 이미 ANN 인덱스를 쓰고 있었던 셈입니다.

In [1]:
import os

import chromadb

VECTORSTORE_DIR = os.path.join("..", "vectorstore")
client = chromadb.PersistentClient(path=VECTORSTORE_DIR)
collection = client.get_collection("langchain")
print(f"지금까지 실습에 쓴 청크 수: {collection.count()}개")

지금까지 실습에 쓴 청크 수: 73개


이 정도 규모(수십~수백 개)에서는 brute-force든 HNSW든 검색 시간이 둘 다 1ms 안팎이라 체감 차이가 없습니다. 그래서 벡터 개수를 인위적으로 늘려서, 몇천~몇십만 개 규모에서는 실제로 어떤 차이가 나는지 직접 재보겠습니다.

## 벡터 개수를 늘려서 직접 시간을 재보기

실제 문서 대신, 텍스트 없이 숫자로만 이루어진 가상의 임베딩 벡터를 대량으로 만들어서 두 방식의 검색 속도를 비교합니다. 임베딩 차원은 실습 편의상 256으로 줄였습니다(실제 임베딩 모델은 보통 512~4096차원).

In [2]:
import time

import numpy as np


def make_synthetic_vectors(n, dim, seed=42):
    # 텍스트 없이 숫자로만 이루어진 가짜 임베딩 n개를 만듭니다 (단위벡터로 정규화).
    rng = np.random.default_rng(seed)
    vecs = rng.normal(size=(n, dim)).astype("float32")
    vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs


sample = make_synthetic_vectors(3, dim=256)
print(sample.shape)  # (3, 256) 형태의 가짜 임베딩 3개

(3, 256)


In [3]:
def compare_search_speed(n, dim=256, n_queries=50):
    """벡터 n개 중에서 brute-force와 Chroma(HNSW)로 top-5를 찾는 평균 시간을 비교합니다."""
    vectors = make_synthetic_vectors(n, dim)
    queries = make_synthetic_vectors(n_queries, dim, seed=7)

    # brute-force: 질문마다 벡터 전체와 내적을 계산해 상위 5개를 뽑음
    t0 = time.perf_counter()
    for q in queries:
        sims = vectors @ q
        np.argpartition(-sims, 5)[:5]
    t_brute = (time.perf_counter() - t0) / n_queries

    # HNSW: Chroma 컬렉션에 넣어 인덱스를 만든 뒤(1회), 같은 질문들로 검색
    tmp_client = chromadb.EphemeralClient()
    tmp_collection = tmp_client.create_collection(f"demo_{n}", metadata={"hnsw:space": "cosine"})
    ids = [str(i) for i in range(n)]
    batch = 5000
    t0 = time.perf_counter()
    for i in range(0, n, batch):
        tmp_collection.add(ids=ids[i:i + batch], embeddings=vectors[i:i + batch].tolist())
    t_index = time.perf_counter() - t0

    t0 = time.perf_counter()
    for q in queries:
        tmp_collection.query(query_embeddings=[q.tolist()], n_results=5)
    t_hnsw = (time.perf_counter() - t0) / n_queries

    print(
        f"N={n:>7} | brute-force 평균 {t_brute * 1000:7.3f}ms | "
        f"HNSW 평균 {t_hnsw * 1000:7.3f}ms | 인덱스 구축(1회) {t_index:6.2f}s | "
        f"HNSW가 {t_brute / t_hnsw:5.2f}배"
    )

청크 1,000개(오늘 실습 데이터의 약 14배)와 100,000개, 두 규모로 비교합니다. 100,000개 인덱스를 만드는 데는 30~40초 정도 걸립니다.

In [4]:
for n in [1000, 100_000]:
    compare_search_speed(n)

N=   1000 | brute-force 평균   0.045ms | HNSW 평균   0.438ms | 인덱스 구축(1회)   0.06s | HNSW가  0.10배


N= 100000 | brute-force 평균   2.167ms | HNSW 평균   1.143ms | 인덱스 구축(1회)  22.33s | HNSW가  1.90배


## 결과 해석

- **1,000개 규모**에서는 오히려 HNSW가 더 느립니다. 그래프를 타고 내려가는 과정 자체에 오버헤드가 있어서, 벡터 수가 적을 땐 그냥 다 비교하는(brute-force) 쪽이 더 빠릅니다.
- **100,000개 규모**부터는 HNSW가 앞서기 시작합니다. 인덱스를 만드는 비용(수십 초)은 처음 한 번만 내면 되고, 이후 검색은 계속 그 빨라진 상태를 재사용합니다.
- 실제 프로덕션 환경은 벡터가 수백만~수억 개 규모입니다. 이 시점에서 brute-force는 사실상 쓸 수 없을 만큼 느려지지만, HNSW는 여전히 빠릅니다. 오늘 실습 데이터(청크 73개)에서는 사실상 의미 없는 최적화입니다.
- 다만 ANN은 "거의 정확한" 근사치이지 100% 정확한 최근접 이웃은 아닙니다(recall이 1보다 작을 수 있음). HNSW 외에도 IVF, PQ 등 다양한 인덱스 방식이 있으며, 원리는 PPT에서 다룹니다.